In [ ]:
import sys
sys.path.append('/kaggle/input/icr-utils')

import os, pickle, gc, warnings, copy, shutil
from glob import glob
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.metrics import log_loss

from utils import load_pickle, pred_multi_to_single, load_model, get_feat_name \
    , get_ft_woe_mappings, woe_map_fts

ORI_DIR = '/kaggle/input/icr-identify-age-related-conditions'
FE_DIR = '/kaggle/input/icr-fe-v3'
TR_DIRS = {
    0: '/kaggle/input/icr-train-lr-d3-fs1-v0'
    , 1: '/kaggle/input/icr-train-lgbm-d3-fs1-v0'
#     , 2: '/kaggle/input/icr-train-xgb-d3-fs1-v0'
}
sys.path.append(FE_DIR)
from feature_engineering import FeatureEngineer

In [ ]:
CONFIGS = {}
for k,v in TR_DIRS.items():
    config = load_pickle(os.path.join(v, 'config.pkl'))
    result = {
        'multi_class': config['MULTI_CLASS']
        , 'imbal_treater': config['IMBAL_TREATER']
        , 'rs': config['RS']
        , 'model_type': config['MODEL_TYPE']
        , 'n_fold': config['N_FOLD']
        , 'n_split': config['N_SPLIT']
        , 'n_param': len(glob(f"{v}/{config['MODEL_TYPE']}/*"))
        , 'fe': load_pickle(os.path.join(v, 'FeatureEngineer.pkl'))
        , 'model_weights': load_pickle(os.path.join(v, 'model_weights.pkl'))
        , 'unweighted_total_oof': load_pickle(os.path.join(v, 'unweighted_total_oof.pkl'))
        , 'weighted_total_oof': load_pickle(os.path.join(v, 'weighted_total_oof.pkl'))
        , 'tr_dir': v
        , 'woe_mapping': config.get('WOE_MAPPING', False)
    }
    CONFIGS[k] = result.copy()

In [ ]:
df_train = pd.read_csv(os.path.join(ORI_DIR, 'train.csv'))
df_test = pd.read_csv(os.path.join(ORI_DIR, 'test.csv'))

In [ ]:
def balanced_log_loss(y_true, y_pred):
    nc = np.bincount(y_true)
    return log_loss(y_true, y_pred, sample_weight = 1/nc[y_true], eps=1e-15)

y_true = df_train.loc[:, 'Class']
y_true = (y_true > 0).astype(int)

# Inference

In [ ]:
final_pred = []
total_weight = 0
agg_oof = np.zeros((len(y_true),2))
for _, config in CONFIGS.items():
    n_model = np.prod(config['model_weights'].shape)
    n_param = config['n_param']
    n_split = config['n_split']
    n_fold = config['n_fold']
    model_type = config['model_type']
    tr_dir = config['tr_dir']
    FE = config['fe']
    multi_class = config['multi_class']
    model_weights = config['model_weights']
    unweighted_oof = config['unweighted_total_oof']
    weighted_oof = config['weighted_total_oof']
    agg_oof += unweighted_oof
    target_col_name = 'Multiclass' if multi_class else 'Class'
    woe_mapping = config['woe_mapping']
    if woe_mapping:
        woe = load_pickle(f'{tr_dir}/WoE_{target_col_name}.pkl')
    
    print('#'*20)
    print(config['model_type'])
    print('#'*20)
    print(f"N-fold={n_fold}")
    print(f"N-split={n_split}")
    print(f"N-param={n_param}")
    print(f"====> Total model={n_model}")
    
    df_test_fe = FE.transform(df_test.copy(), is_training=False)
    
    for p_idx in range(config['n_param']):
        print(f'{p_idx}', end=', ')
        # Get feature names
        model_path = f'{tr_dir}/{model_type}/{model_type}_{p_idx}/split_0/fold_0'
        model = load_model(model_type, model_path)
        features = get_feat_name(model_type, model)
    
        if woe_mapping:
            woe_mappings = get_ft_woe_mappings(woe, features)
            df_test_final = woe_map_fts(df_test_fe, woe_mappings, features)
        else:
            df_test_final = df_test_fe.copy()
        
        for s_id in range(n_split):
            _ = gc.collect()
            for f_id in range(n_fold):

                # Get model
                model_path = f'{tr_dir}/{model_type}/{model_type}_{p_idx}/split_{s_id}/fold_{f_id}'
                model = load_model(model_type, model_path)

                # Inference
                pred = model.predict_proba(df_test_final.loc[:, features].to_numpy())
                pred = pred_multi_to_single(pred, multi_class=multi_class)
                fold_weight = model_weights[p_idx, s_id, f_id]
                final_pred.append(pred * fold_weight)
                
                # Update weight
                total_weight += fold_weight
    print()
    print(f'Set OOF loss: {balanced_log_loss(y_true, weighted_oof)}')
    print(f'Agg OOF loss: {balanced_log_loss(y_true, agg_oof/total_weight)}')
    print()
                
final_pred = np.sum(final_pred, axis=0)
final_pred = final_pred / total_weight
final_pred

# Submission

In [ ]:
sample_submission = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/sample_submission.csv')
sample_submission[['class_0','class_1']] = final_pred
sample_submission.to_csv('submission.csv',index=False)
sample_submission